# DINO_OBB

In [13]:
import pickle
import numpy as np
import math
import os

from PIL import Image, ImageDraw

In [2]:
def cal_line_length(point1, point2):
    """Calculate the length of line.

    Args:
        point1 (List): [x,y]
        point2 (List): [x,y]

    Returns:
        length (float)
    """
    return math.sqrt(
        math.pow(point1[0] - point2[0], 2) +
        math.pow(point1[1] - point2[1], 2))

def get_best_begin_point_single(coordinate):
    """Get the best begin point of the single polygon.

    Args:
        coordinate (List): [x1, y1, x2, y2, x3, y3, x4, y4, score]

    Returns:
        reorder coordinate (List): [x1, y1, x2, y2, x3, y3, x4, y4, score]
    """
    x1, y1, x2, y2, x3, y3, x4, y4, score = coordinate
    xmin = min(x1, x2, x3, x4)
    ymin = min(y1, y2, y3, y4)
    xmax = max(x1, x2, x3, x4)
    ymax = max(y1, y2, y3, y4)
    combine = [[[x1, y1], [x2, y2], [x3, y3], [x4, y4]],
               [[x2, y2], [x3, y3], [x4, y4], [x1, y1]],
               [[x3, y3], [x4, y4], [x1, y1], [x2, y2]],
               [[x4, y4], [x1, y1], [x2, y2], [x3, y3]]]
    dst_coordinate = [[xmin, ymin], [xmax, ymin], [xmax, ymax], [xmin, ymax]]
    force = 100000000.0
    force_flag = 0
    for i in range(4):
        temp_force = cal_line_length(combine[i][0], dst_coordinate[0]) \
                     + cal_line_length(combine[i][1], dst_coordinate[1]) \
                     + cal_line_length(combine[i][2], dst_coordinate[2]) \
                     + cal_line_length(combine[i][3], dst_coordinate[3])
        if temp_force < force:
            force = temp_force
            force_flag = i
    if force_flag != 0:
        pass
    return np.hstack(
        (np.array(combine[force_flag]).reshape(8), np.array(score)))


def get_best_begin_point(coordinates):
    """Get the best begin points of polygons.

    Args:
        coordinate (ndarray): shape(n, 9).

    Returns:
        reorder coordinate (ndarray): shape(n, 9).
    """
    coordinates = list(map(get_best_begin_point_single, coordinates.tolist()))
    coordinates = np.array(coordinates)
    return coordinates

def obb2poly_np_le90(obboxes):
    """Convert oriented bounding boxes to polygons.

    Args:
        obbs (ndarray): [x_ctr,y_ctr,w,h,angle,score]

    Returns:
        polys (ndarray): [x0,y0,x1,y1,x2,y2,x3,y3,score]
    """
    try:
        center, w, h, theta, score = np.split(obboxes, (2, 3, 4, 5), axis=-1)
    except:  # noqa: E722
        results = np.stack([0., 0., 0., 0., 0., 0., 0., 0., 0.], axis=-1)
        return results.reshape(1, -1)
    Cos, Sin = np.cos(theta), np.sin(theta)
    vector1 = np.concatenate([w / 2 * Cos, w / 2 * Sin], axis=-1)
    vector2 = np.concatenate([-h / 2 * Sin, h / 2 * Cos], axis=-1)
    point1 = center - vector1 - vector2
    point2 = center + vector1 - vector2
    point3 = center + vector1 + vector2
    point4 = center - vector1 + vector2
    polys = np.concatenate([point1, point2, point3, point4, score], axis=-1)
    polys = get_best_begin_point(polys)
    return polys


In [3]:
def get_gt_annos(gt_dicts) :
    '''
    args:
        #txt_file: dota-style json filw where bbox format of dota style is x1, y1, x2, y2, x3, y3, x4, y4
        #class_names: list of class names in order of predcitions
        #imgs: list of image names
        #ROOT: path of annotation files
    return:
        gt_annos: dictionary of [image_id, x1, y1, x2, y2, x3, y3, x4, y4] by class
    '''
    gt_annos = {}
    for class_name in ['pylon', 'powerline'] :
        gt_annos[class_name] = []

    for n, gt in enumerate(gt_dicts) :
        poly_gts = obb2poly_np_le90(gt)

        for poly_gt in poly_gts :
            cls = poly_gt[-1]
            if cls == 0 :
                class_name = 'pylon'
            else :
                class_name = 'powerline'
            polygon = poly_gt[:-1]
            gt_annos[class_name].append([n] + list(polygon))

    for class_name in ['pylon', 'powerline']:
        gt_annos[class_name] = np.array(gt_annos[class_name])
        
    return gt_annos

In [4]:
def get_sorted_preds(pred_dicts, score_th =0.5) :
    '''
    args:
        pred_dicts: dictionary of result of `inference_detector` api in mmrotate with key of image_name
        class_names: list of class names in order of predcitions
        imgs: list of image names
    retrun:
        sorted_preds: dictionary of lists of [score, [x1, y1, x2, y2, x3, y3, x4, y4], image_id] sorted by scores according to the class
    '''

    preds = {}
    sorted_preds = {}
    for class_name in ['pylon', 'powerline']  :
        preds[class_name] = []
        sorted_preds[class_name] = []

    for img_id, output  in enumerate(pred_dicts) :
        clses_per_file = output.numpy()[:,-1] # 300,
        preds_per_file = obb2poly_np_le90(output.numpy()[:,:-1]) # 300, 7
        for n, pred_per_file in enumerate(preds_per_file) :
            cls = clses_per_file[n]
            if cls == 0 :
                class_name = 'pylon'
            else :
                class_name = 'powerline'
            score = pred_per_file[-1]
            if score > score_th :
                polygon = list(pred_per_file[:-1])
                preds[class_name].append([score] + polygon + [img_id])

    for class_name in ['pylon', 'powerline']  :
        preds[class_name] = np.array(preds[class_name])
        confidence = preds[class_name][:,0]
        sorted_ind = np.argsort(-confidence)
        sorted_preds[class_name] = preds[class_name][sorted_ind, :]

    return sorted_preds

In [5]:
from shapely.geometry import Polygon

def compute_iou(poly1, poly2):
    """
    Compute the IoU between two polygons.
    
    Args:
    poly1, poly2: List of coordinates [x1, y1, x2, y2, x3, y3, x4, y4]
    
    Returns:
    IoU value (float)
    """
    # Convert lists to shapely polygons
    polygon1 = Polygon([(poly1[i], poly1[i+1]) for i in range(0, len(poly1), 2)])
    polygon2 = Polygon([(poly2[i], poly2[i+1]) for i in range(0, len(poly2), 2)])
    
    # Compute intersection and union
    intersection = polygon1.intersection(polygon2).area
    union = polygon1.union(polygon2).area

    # Compute IoU
    iou = intersection / union if union > 0 else 0
    return iou

In [6]:
def get_tp (sorted_preds, gt_annos, iou_th) :
    '''
    args
        sorted_preds: list of [score, XYs, image_id] sorted by scores
        gt_annos: list of [image_id, XYs]
    return
        tp: list of correct ones
        fp: list of wrong ones
    '''
    tp = np.zeros(sorted_preds.shape[0])
    fp = np.zeros(sorted_preds.shape[0])
    gt_annos_for_cf = np.copy(gt_annos)
    for i, sorted_pred in enumerate(sorted_preds) :
        img_id = int(sorted_pred[-1])
        pred_poly = sorted_pred[1:-1]
        #print (cx, cy)
        for j in np.where(gt_annos_for_cf[:,0] == img_id)[0] :
            # get_iou
            gt_poly = gt_annos_for_cf[j,1:]
            iou = compute_iou(pred_poly, gt_poly)
            if iou >= iou_th :
                tp[i] = 1
                gt_annos_for_cf[j] = np.array([-1,-1,-1,-1,-1,-1,-1,-1,-1])
                continue
        if not tp[i] == 1 :
            fp[i] = 1 
    return tp, fp

In [7]:
def get_ap(rec, prec, use_07_metric=False):
    """ 
    Compute VOC AP given precision and recall.
    If use_07_metric is true, uses the
    VOC 07 11 point method (default:False).
    """
    if use_07_metric:
        # 11 point metric
        ap = 0.
        for t in np.arange(0., 1.1, 0.1):
            if np.sum(rec >= t) == 0:
                p = 0
            else:
                p = np.max(prec[rec >= t])
            ap = ap + p / 11.
    else:
        # correct AP calculation
        # first append sentinel values at the end
        mrec = np.concatenate(([0.], rec, [1.]))
        mpre = np.concatenate(([0.], prec, [0.]))

        # compute the precision envelope
        for i in range(mpre.size - 1, 0, -1):
            mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])

        # to calculate area under PR curve, look for points
        # where X axis (recall) changes value
        i = np.where(mrec[1:] != mrec[:-1])[0]

        # and sum (\Delta recall) * prec
        ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    return ap

In [61]:
with open('./exp/nmg/fusion_0.0_1.0_e500/result/OBB/gt.pkl', 'rb') as f:
    gts = pickle.load(f)
    
with open('./exp/nmg/fusion_0.0_1.0_e500/result/OBB/pred.pkl', 'rb') as f:
    preds = pickle.load(f)
    
sorted_preds = get_sorted_preds(preds, 0.001) # [score, XYs, image_id] 
gt_annos = get_gt_annos(gts) # [image_id, XYs]

In [62]:
class_name = 'pylon'
tp, fp = get_tp(sorted_preds[class_name], gt_annos[class_name], 0.5)
npos = len(gt_annos[class_name])


for score_th in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]: #, 0.3, 0.4, 0.5] :
    th_idx = np.where(sorted_preds['pylon'][:,0] > score_th)[0]#[-1] + 1
    recall = tp[th_idx].sum()/npos
    precision = tp[th_idx].sum()/th_idx.shape[0]
    f1 = 2 / (1/recall + 1/precision)
    print (score_th, ', {}, {:.2f}, {:.2f}, {:.2f}'.format(th_idx[-1]+1, 100*precision, 100*recall, 100*f1))

# compute precision recall
fps = np.cumsum(fp)
tps = np.cumsum(tp)
rec = tps / float(npos)
use_07_metric = False
# avoid divide by zero in case the first detection matches a difficult
# ground truth
prec = tps / np.maximum(tps + fps, np.finfo(np.float64).eps)
ap = get_ap(rec, prec, use_07_metric)
print (ap*100)
# 87.5

0.1 , 481, 33.68, 76.42, 46.75
0.2 , 272, 54.41, 69.81, 61.16
0.3 , 217, 62.67, 64.15, 63.40
0.4 , 188, 67.55, 59.91, 63.50
0.5 , 163, 71.17, 54.72, 61.87
0.6 , 136, 75.74, 48.58, 59.20
0.7 , 121, 81.82, 46.70, 59.46
0.8 , 104, 83.65, 41.04, 55.06
0.9 , 86, 82.56, 33.49, 47.65
61.282852157527955


In [63]:
class_name = 'powerline'
tp, fp = get_tp(sorted_preds[class_name], gt_annos[class_name], 0.5)
npos = len(gt_annos[class_name])


for score_th in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]: #, 0.3, 0.4, 0.5] :
    th_idx = np.where(sorted_preds['powerline'][:,0] > score_th)[0]#[-1] + 1
    recall = tp[th_idx].sum()/npos
    precision = tp[th_idx].sum()/th_idx.shape[0]
    f1 = 2 / (1/recall + 1/precision)
    print (score_th, ', {}, {:.2f}, {:.2f}, {:.2f}'.format(th_idx[-1]+1, 100*precision, 100*recall, 100*f1))

# compute precision recall
fps = np.cumsum(fp)
tps = np.cumsum(tp)
rec = tps / float(npos)
use_07_metric = False
# avoid divide by zero in case the first detection matches a difficult
# ground truth
prec = tps / np.maximum(tps + fps, np.finfo(np.float64).eps)
ap = get_ap(rec, prec, use_07_metric)
print (ap*100)
# 75.1 // Max_Recall: 79.09 / 83.o95

0.1 , 513, 44.64, 67.16, 53.63
0.2 , 365, 60.00, 64.22, 62.04
0.3 , 309, 67.64, 61.29, 64.31
0.4 , 268, 75.37, 59.24, 66.34
0.5 , 236, 80.08, 55.43, 65.51
0.6 , 211, 85.31, 52.79, 65.22
0.7 , 191, 86.39, 48.39, 62.03
0.8 , 169, 87.57, 43.40, 58.04
0.9 , 135, 89.63, 35.48, 50.84
60.20955630212194


# Visualization

In [64]:
def obb2poly_np_le90(obboxes):
    """Convert oriented bounding boxes to polygons.

    Args:
        obbs (ndarray): [x_ctr,y_ctr,w,h,angle,score]

    Returns:
        polys (ndarray): [x0,y0,x1,y1,x2,y2,x3,y3,score]
    """
    try:
        center, w, h, theta, score = np.split(obboxes, (2, 3, 4, 5), axis=-1)
    except:  # noqa: E722
        results = np.stack([0., 0., 0., 0., 0., 0., 0., 0., 0.], axis=-1)
        return results.reshape(1, -1)
    Cos, Sin = np.cos(theta), np.sin(theta)
    vector1 = np.concatenate([w / 2 * Cos, w / 2 * Sin], axis=-1)
    vector2 = np.concatenate([-h / 2 * Sin, h / 2 * Cos], axis=-1)
    point1 = center - vector1 - vector2
    point2 = center + vector1 - vector2
    point3 = center + vector1 + vector2
    point4 = center - vector1 + vector2
    polys = np.concatenate([point1, point2, point3, point4, score], axis=-1)
    polys = get_best_begin_point(polys)
    return polys

In [66]:
import glob
filenames = [os.path.split(f)[1] for f in glob.glob('/nas2/YJ/DATA/NMG/3D_RAW_7/cropped1024/test/*.las')]

ROOT = '/nas2/YJ/DATA/NMG/2D_RAW/MSD/cropped1024/test/images/'

GT_ROOT = '/nas2/YJ/DATA/NMG/3D_RAW_7/cropped1024/test/visualization/'
DEST = 'exp/nmg/fusion_0.0_1.0_e500/result/OBB/comparision_gt'
os.makedirs(DEST, exist_ok=True)

pylon_score_th = 0.4
span_score_th = 0.4

rhino_pylon_score_th = 0.7
rhino_span_score_th = 0.4

for i, filename in enumerate(filenames) :
    filename = filename.replace('las', 'png')
    img_path = os.path.join(ROOT, filename)

    pred = preds[i]
    pred_labels = pred[:,-1]
    pred_boxes = pred[:,:-1]
    pred_poly = obb2poly_np_le90(pred_boxes)

    pylons = []
    spans = []
    for n, pred_label in enumerate(pred_labels) :
        score = pred_poly[n][-1]
        poly = pred_poly[n][:-1]
        if pred_label == 0 and score > pylon_score_th :
            #pylons.append(np.append(poly, pred_label))
            pylons.append(poly)
        if pred_label == 1 and score > span_score_th :
            #spans.append(np.append(poly, pred_label))
            spans.append(poly)

    im = Image.open(img_path)
    draw = ImageDraw.Draw(im)
    
    for pylon in pylons :
        xy = list(pylon)
        draw.polygon(xy, outline="pink", width=4)
    
    for span in spans :
        xy = list(span)
        draw.polygon(xy, outline="white", width=4)
        
    # GTs
    # DINO_OBB preds
    gt = gts[i]
    gt_poly = obb2poly_np_le90(gt)
    
    gt_pylons = []
    gt_spans = []
    for poly in gt_poly :
        label = poly[-1]
        poly = poly[:-1]
        if label == 0 :
            gt_pylons.append(poly)
        if label == 1 :
            gt_spans.append(poly)
            
    gt_im = Image.open(img_path)
    gt_draw = ImageDraw.Draw(gt_im)

    for pylon in gt_pylons :
        xy = list(pylon)
        gt_draw.polygon(xy, outline="pink", width=4)
    
    for span in gt_spans :
        xy = list(span)
        gt_draw.polygon(xy, outline="white", width=4)

    w,h = im.size
    #print (img, im.size, gt_im.size)
    new_im = Image.new('RGB', (w+w, h), 'white')
    new_im.paste(gt_im, (0,0))
    new_im.paste(im, (w,0))

    new_im.save(os.path.join(DEST, filename))
print ('Done!!')

Done!!


In [67]:
import glob
filenames = [os.path.split(f)[1] for f in glob.glob('/nas2/YJ/DATA/NMG/3D_RAW_7/cropped1024/test/*.las')]

ROOT = '/nas2/YJ/DATA/NMG/2D_RAW/MSD/cropped1024/test/images/'

GT_ROOT = '/nas2/YJ/DATA/NMG/3D_RAW_7/cropped1024/test/visualization/'
DEST = 'exp/nmg/fusion_0.0_1.0_e500/result/OBB/comparision_all'
os.makedirs(DEST, exist_ok=True)

with open ('/nas2/YJ/git/RHINO/work_dirs/rhino_MSD_shape1920/preds.pkl', 'br') as f :
    rhino_preds = pickle.load(f)

pylon_score_th = 0.4
span_score_th = 0.4

rhino_pylon_score_th = 0.7
rhino_span_score_th = 0.4

for i, filename in enumerate(filenames) :
    filename = filename.replace('las', 'png')
    img_path = os.path.join(ROOT, filename)
    
    # DINO_OBB preds
    gt = gts[i]
    gt_poly = obb2poly_np_le90(gt)

    pred = preds[i]
    pred_labels = pred[:,-1]
    pred_boxes = pred[:,:-1]
    pred_poly = obb2poly_np_le90(pred_boxes)

    pylons = []
    spans = []
    for n, pred_label in enumerate(pred_labels) :
        score = pred_poly[n][-1]
        poly = pred_poly[n][:-1]
        if pred_label == 0 and score > pylon_score_th :
            #pylons.append(np.append(poly, pred_label))
            pylons.append(poly)
        if pred_label == 1 and score > span_score_th :
            #spans.append(np.append(poly, pred_label))
            spans.append(poly)

    im = Image.open(img_path)
    draw = ImageDraw.Draw(im)
    
    for pylon in pylons :
        xy = list(pylon)
        draw.polygon(xy, outline="pink", width=4)
    
    for span in spans :
        xy = list(span)
        draw.polygon(xy, outline="white", width=4)
        
    # Rhino preds
    rhino_pred_pylon, rhino_pred_span = rhino_preds[filename]
    rhino_pylons = obb2poly_np_le90(rhino_pred_pylon)
    rhino_spans = obb2poly_np_le90(rhino_pred_span)
    
    rhino_im = Image.open(img_path)
    rhino_draw = ImageDraw.Draw(rhino_im)

    for pylon in rhino_pylons :
        score = pylon[-1]
        if score > rhino_pylon_score_th :
            xy = list(pylon[:-1])
            rhino_draw.polygon(xy, outline="pink", width=4)
    
    for span in rhino_spans :
        score = span[-1]
        if score > rhino_span_score_th :
            xy = list(span[:-1])
            rhino_draw.polygon(xy, outline="white", width=4)

    gt_im = Image.open(os.path.join(GT_ROOT, filename.replace('.png', '_annos.png')))
    
    w,h = im.size
    #print (img, im.size, gt_im.size)
    new_im = Image.new('RGB', (w+w+w, h), 'white')
    new_im.paste(gt_im, (0,0))
    new_im.paste(im, (w,0))
    new_im.paste(rhino_im, (w+w,0))

    new_im.save(os.path.join(DEST, filename))
print ('Done!!')

Done!!
